<a href="https://colab.research.google.com/github/AscanOfficiel2/MDT-TM_CRC_Microbiome-Framework/blob/main/LOCO_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ============================================================
# CRC MICROBIOME — LEAVE-ONE-COHORT-OUT VALIDATION
# SVM | CLR + Batch-Corrected v2 Matrix
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    confusion_matrix
)

RANDOM_STATE = 42
CLASSES = ["Healthy", "Adenoma", "Cancer"]

# ============================================================
# 1. LOAD DATA
# ============================================================

X_df = pd.read_csv(
    "/content/TPM_clr_batch_corrected_v2.csv",
    index_col=0
)

meta = pd.read_csv("/content/CRC_metadata_aligned_bact.csv")

# Identify cohort column
cohort_candidates = ["Project", "Cohort", "Cohort_ID"]

cohort_col = next(
    (c for c in cohort_candidates if c in meta.columns),
    None
)

if cohort_col is None:
    raise ValueError(
        "No cohort column found. Metadata must contain "
        "'Project', 'Cohort', or 'Cohort_ID'. "
        "Merge sample-to-project assignments first."
    )

# ============================================================
# 2. ALIGN SAMPLES
# ============================================================

X_df.index = X_df.index.astype(str).str.strip()
meta["Sample_ID"] = meta["Sample_ID"].astype(str).str.strip()

if X_df.index.has_duplicates:
    raise ValueError("Duplicate sample IDs in abundance matrix.")

if meta["Sample_ID"].duplicated().any():
    raise ValueError("Duplicate sample IDs in metadata.")

common = X_df.index.intersection(meta["Sample_ID"])

meta = (
    meta.set_index("Sample_ID")
        .loc[common]
        .copy()
)

X_df = X_df.loc[common]

meta = meta.dropna(subset=["Group", cohort_col])
X_df = X_df.loc[meta.index]

X_df = X_df.apply(pd.to_numeric, errors="coerce")

print("Samples:", len(meta))
print("Features:", X_df.shape[1])
print("Cohorts:", meta[cohort_col].nunique())

print("\nClinical group distribution:")
print(meta["Group"].value_counts())

cohort_distribution = pd.crosstab(
    meta[cohort_col],
    meta["Group"]
).reindex(columns=CLASSES, fill_value=0)

cohort_distribution["Total"] = cohort_distribution.sum(axis=1)

print("\nCohort distribution:")
print(cohort_distribution)

cohort_distribution.to_csv(
    "LOCO_Cohort_Distribution.csv"
)

# ============================================================
# 3. LEAVE-ONE-COHORT-OUT VALIDATION
# ============================================================

results = []
predictions = []

for held_out in sorted(meta[cohort_col].unique()):

    train_mask = meta[cohort_col] != held_out
    test_mask = meta[cohort_col] == held_out

    X_train = X_df.loc[train_mask].values
    X_test = X_df.loc[test_mask].values

    y_train = meta.loc[train_mask, "Group"].values
    y_test = meta.loc[test_mask, "Group"].values

    if len(np.unique(y_train)) < 3:
        print(
            f"Skipping {held_out}: "
            "training data lack one or more classes."
        )
        continue

    # Original SVM configuration
    model = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "svm",
            SVC(
                kernel="rbf",
                C=10,
                gamma="scale",
                probability=True,
                class_weight="balanced",
                random_state=RANDOM_STATE
            )
        )
    ])

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)

    # Align probability columns to fixed class order
    prob_df = pd.DataFrame(
        y_prob,
        columns=model.classes_,
        index=meta.index[test_mask]
    ).reindex(columns=CLASSES)

    # AUROC: only for evaluable one-vs-rest classes
    auc_values = {}

    for cls in CLASSES:
        binary_true = (y_test == cls).astype(int)

        if len(np.unique(binary_true)) == 2:
            auc_values[cls] = roc_auc_score(
                binary_true,
                prob_df[cls].values
            )
        else:
            auc_values[cls] = np.nan

    available_auc = [
        v for v in auc_values.values()
        if np.isfinite(v)
    ]

    macro_auc = (
        np.mean(available_auc)
        if available_auc else np.nan
    )

    # Metrics
    row = {
        "Held_out_cohort": held_out,
        "N_test": len(y_test),
        "N_Healthy": sum(y_test == "Healthy"),
        "N_Adenoma": sum(y_test == "Adenoma"),
        "N_Cancer": sum(y_test == "Cancer"),
        "Accuracy": accuracy_score(y_test, y_pred),
        "BalancedAccuracy": balanced_accuracy_score(
            y_test, y_pred
        ),
        "MacroF1": f1_score(
            y_test,
            y_pred,
            labels=CLASSES,
            average="macro",
            zero_division=0
        ),
        "MCC": matthews_corrcoef(y_test, y_pred),
        "MacroAUROC_evaluable": macro_auc,
        "AUROC_Healthy": auc_values["Healthy"],
        "AUROC_Adenoma": auc_values["Adenoma"],
        "AUROC_Cancer": auc_values["Cancer"]
    }

    results.append(row)

    pred_fold = pd.DataFrame({
        "Sample_ID": meta.index[test_mask],
        "Held_out_cohort": held_out,
        "True_Group": y_test,
        "Predicted_Group": y_pred
    })

    for cls in CLASSES:
        pred_fold[f"P_{cls}"] = prob_df[cls].values

    predictions.append(pred_fold)

    print(
        f"{held_out}: n={len(y_test)}, "
        f"Balanced accuracy={row['BalancedAccuracy']:.3f}, "
        f"MCC={row['MCC']:.3f}"
    )

# ============================================================
# 4. SAVE COHORT-SPECIFIC RESULTS
# ============================================================

results_df = pd.DataFrame(results)
pred_df = pd.concat(predictions, ignore_index=True)

results_df.to_csv(
    "LOCO_SVM_Performance_ByCohort.csv",
    index=False
)

pred_df.to_csv(
    "LOCO_SVM_OutOfCohort_Predictions.csv",
    index=False
)

# ============================================================
# 5. POOLED OUT-OF-COHORT PERFORMANCE
# ============================================================

y_true = pred_df["True_Group"].values
y_pred = pred_df["Predicted_Group"].values

pooled = {
    "N": len(y_true),
    "Accuracy": accuracy_score(y_true, y_pred),
    "BalancedAccuracy": balanced_accuracy_score(
        y_true, y_pred
    ),
    "MacroF1": f1_score(
        y_true,
        y_pred,
        labels=CLASSES,
        average="macro",
        zero_division=0
    ),
    "MCC": matthews_corrcoef(y_true, y_pred)
}

for cls in CLASSES:
    binary = (y_true == cls).astype(int)

    pooled[f"AUROC_{cls}"] = (
        roc_auc_score(
            binary,
            pred_df[f"P_{cls}"].values
        )
        if len(np.unique(binary)) == 2
        else np.nan
    )

pd.DataFrame([pooled]).to_csv(
    "LOCO_SVM_Pooled_Performance.csv",
    index=False
)

print("\nPooled performance:")
print(pd.Series(pooled))

# ============================================================
# 6. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=CLASSES
)

fig, ax = plt.subplots(figsize=(5, 4.5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASSES,
    yticklabels=CLASSES,
    cbar=False,
    ax=ax
)

ax.set_xlabel("Predicted clinical group")
ax.set_ylabel("Observed clinical group")
ax.set_title("SVM Leave-One-Cohort-Out Validation")

plt.tight_layout()
plt.savefig(
    "LOCO_SVM_ConfusionMatrix.png",
    dpi=600,
    bbox_inches="tight"
)
plt.show()

print("\nLOCO validation completed.")


In [ ]:

# ================================================================
# CRC MICROBIOME — CLR-ONLY LOCO VALIDATION
# Comparison with Globally Batch-Corrected LOCO
#
# Original TPM -> CLR -> SVM -> Leave-One-Cohort-Out
#
# Same samples, features and SVM as previous analysis
# ================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# ================================================================
# 0. SETTINGS
# ================================================================

RANDOM_STATE = 42
PSEUDOCOUNT = 1e-6

CLASSES = ["Healthy", "Adenoma", "Cancer"]

TPM_FILE = "TPM_raw_filtered_bact.csv"
META_FILE = "CRC_metadata_aligned_bact.csv"
V2_FILE = "TPM_clr_batch_corrected_v2.csv"

OUTPUT_DIR = "LOCO_CLRonly_Results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ================================================================
# 1. LOAD DATA
# ================================================================

tpm = pd.read_csv(TPM_FILE, index_col=0)
meta = pd.read_csv(META_FILE)
v2 = pd.read_csv(V2_FILE, index_col=0)

# Clean identifiers
tpm.index = tpm.index.astype(str).str.strip()
tpm.columns = tpm.columns.astype(str).str.strip()

v2.index = v2.index.astype(str).str.strip()
v2.columns = v2.columns.astype(str).str.strip()

meta["Sample_ID"] = (
    meta["Sample_ID"].astype(str).str.strip()
)

# Identify cohort column
cohort_candidates = [
    "Project", "Cohort", "Cohort_ID"
]

cohort_col = next(
    (c for c in cohort_candidates if c in meta.columns),
    None
)

if cohort_col is None:
    raise ValueError(
        "Metadata requires a Project or Cohort column."
    )

# Check duplicates
if tpm.index.has_duplicates:
    raise ValueError("Duplicate species in TPM matrix.")

if tpm.columns.has_duplicates:
    raise ValueError("Duplicate sample IDs in TPM matrix.")

if v2.index.has_duplicates:
    raise ValueError("Duplicate sample IDs in v2 matrix.")

if v2.columns.has_duplicates:
    raise ValueError("Duplicate species in v2 matrix.")

if meta["Sample_ID"].duplicated().any():
    raise ValueError("Duplicate metadata sample IDs.")

# ================================================================
# 2. MATCH EXACT ORIGINAL ML SAMPLES AND FEATURES
# ================================================================

samples = v2.index.tolist()
features = v2.columns.tolist()

missing_samples = set(samples) - set(tpm.columns)
missing_features = set(features) - set(tpm.index)

if missing_samples:
    raise ValueError(
        f"Missing {len(missing_samples)} samples in TPM."
    )

if missing_features:
    raise ValueError(
        f"Missing {len(missing_features)} species in TPM."
    )

meta = meta.set_index("Sample_ID")

if not set(samples).issubset(meta.index):
    raise ValueError(
        "Some ML samples are missing from metadata."
    )

# Restrict to exact original feature universe
tpm = tpm.loc[features, samples]
meta = meta.loc[samples].copy()

if meta[["Group", cohort_col]].isna().any().any():
    raise ValueError("Missing clinical group or cohort.")

if not set(meta["Group"]).issubset(set(CLASSES)):
    raise ValueError("Unexpected clinical group detected.")

print("======================================")
print("DATA VERIFICATION")
print("======================================")

print("Samples:", len(samples))
print("Species:", len(features))
print("Cohorts:", meta[cohort_col].nunique())

distribution = pd.crosstab(
    meta[cohort_col],
    meta["Group"]
).reindex(columns=CLASSES, fill_value=0)

distribution["Total"] = distribution.sum(axis=1)

print("\nCohort distribution:")
print(distribution)

distribution.to_csv(
    f"{OUTPUT_DIR}/LOCO_Cohort_Distribution.csv"
)

# ================================================================
# 3. CLR TRANSFORMATION — NO BATCH CORRECTION
# ================================================================

X_raw = tpm.T.apply(
    pd.to_numeric,
    errors="coerce"
)

if X_raw.isna().any().any():
    raise ValueError(
        "Missing or non-numeric TPM values detected."
    )

if (X_raw < 0).any().any():
    raise ValueError(
        "Negative TPM values detected."
    )

# Add pseudocount and log transform
log_values = np.log(
    X_raw.to_numpy(dtype=float) + PSEUDOCOUNT
)

# CLR transformation per sample
clr_values = (
    log_values -
    log_values.mean(axis=1, keepdims=True)
)

X_clr = pd.DataFrame(
    clr_values,
    index=X_raw.index,
    columns=X_raw.columns
)

# Verify CLR row sums
assert np.allclose(
    X_clr.sum(axis=1).values,
    0,
    atol=1e-8
)

X_clr.to_csv(
    f"{OUTPUT_DIR}/TPM_CLRonly_Matched.csv"
)

print("\nCLR transformation completed.")
print("Shape:", X_clr.shape)

# ================================================================
# 4. LOCO VALIDATION
# ================================================================

results = []
predictions = []

cohorts = sorted(meta[cohort_col].unique())

for held_out in cohorts:

    train_mask = meta[cohort_col] != held_out
    test_mask = meta[cohort_col] == held_out

    X_train = X_clr.loc[train_mask].values
    X_test = X_clr.loc[test_mask].values

    y_train = meta.loc[train_mask, "Group"].values
    y_test = meta.loc[test_mask, "Group"].values

    if set(y_train) != set(CLASSES):
        raise ValueError(
            f"Training data for {held_out} "
            "do not contain all three classes."
        )

    # Same SVM configuration as previous LOCO
    model = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "svm",
            SVC(
                kernel="rbf",
                C=10,
                gamma="scale",
                probability=True,
                class_weight="balanced",
                random_state=RANDOM_STATE
            )
        )
    ])

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    probabilities = pd.DataFrame(
        model.predict_proba(X_test),
        columns=model.classes_,
        index=meta.index[test_mask]
    ).reindex(columns=CLASSES)

    # Class-specific AUROC
    auc_scores = {}

    for cls in CLASSES:

        y_binary = (y_test == cls).astype(int)

        if len(np.unique(y_binary)) == 2:
            auc_scores[cls] = roc_auc_score(
                y_binary,
                probabilities[cls]
            )
        else:
            auc_scores[cls] = np.nan

    # Cohort performance
    row = {
        "Held_out_cohort": held_out,
        "N_test": len(y_test),
        "N_Healthy": sum(y_test == "Healthy"),
        "N_Adenoma": sum(y_test == "Adenoma"),
        "N_Cancer": sum(y_test == "Cancer"),

        "Accuracy": accuracy_score(
            y_test, y_pred
        ),

        "BalancedAccuracy": balanced_accuracy_score(
            y_test, y_pred
        ),

        "MacroF1": f1_score(
            y_test,
            y_pred,
            labels=CLASSES,
            average="macro",
            zero_division=0
        ),

        "MCC": matthews_corrcoef(
            y_test, y_pred
        ),

        "AUROC_Healthy": auc_scores["Healthy"],
        "AUROC_Adenoma": auc_scores["Adenoma"],
        "AUROC_Cancer": auc_scores["Cancer"]
    }

    results.append(row)

    fold_predictions = pd.DataFrame({
        "Sample_ID": meta.index[test_mask],
        "Held_out_cohort": held_out,
        "True_Group": y_test,
        "Predicted_Group": y_pred
    })

    for cls in CLASSES:
        fold_predictions[f"P_{cls}"] = (
            probabilities[cls].values
        )

    predictions.append(fold_predictions)

    print(
        f"{held_out}: "
        f"n={len(y_test)}, "
        f"BA={row['BalancedAccuracy']:.3f}, "
        f"MCC={row['MCC']:.3f}"
    )

# ================================================================
# 5. SAVE COHORT RESULTS
# ================================================================

results_df = pd.DataFrame(results)

pred_df = pd.concat(
    predictions,
    ignore_index=True
)

results_df.to_csv(
    f"{OUTPUT_DIR}/LOCO_CLRonly_ByCohort.csv",
    index=False
)

pred_df.to_csv(
    f"{OUTPUT_DIR}/LOCO_CLRonly_Predictions.csv",
    index=False
)

# ================================================================
# 6. POOLED OUT-OF-COHORT PERFORMANCE
# ================================================================

y_true = pred_df["True_Group"].values
y_pred = pred_df["Predicted_Group"].values

pooled = {
    "N": len(y_true),

    "Accuracy": accuracy_score(
        y_true, y_pred
    ),

    "BalancedAccuracy": balanced_accuracy_score(
        y_true, y_pred
    ),

    "MacroF1": f1_score(
        y_true,
        y_pred,
        labels=CLASSES,
        average="macro",
        zero_division=0
    ),

    "MCC": matthews_corrcoef(
        y_true, y_pred
    )
}

for cls in CLASSES:

    binary = (y_true == cls).astype(int)

    pooled[f"AUROC_{cls}"] = roc_auc_score(
        binary,
        pred_df[f"P_{cls}"].values
    )

pooled_df = pd.DataFrame([pooled])

pooled_df.to_csv(
    f"{OUTPUT_DIR}/LOCO_CLRonly_Pooled.csv",
    index=False
)

print("\n======================================")
print("POOLED LOCO RESULTS")
print("======================================")
print(pooled_df.round(4).to_string(index=False))

# ================================================================
# 7. POOLED CONFUSION MATRIX
# ================================================================

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=CLASSES
)

fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=CLASSES
).plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=False
)

ax.set_title(
    "SVM LOCO Validation — CLR Only",
    fontsize=15
)

ax.set_xlabel("Predicted clinical group")
ax.set_ylabel("Observed clinical group")

plt.tight_layout()

plt.savefig(
    f"{OUTPUT_DIR}/LOCO_CLRonly_ConfusionMatrix.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

# ================================================================
# 8. POOLED ROC CURVES
# ================================================================

fig, ax = plt.subplots(figsize=(6, 5))

for cls in CLASSES:

    binary = (y_true == cls).astype(int)
    scores = pred_df[f"P_{cls}"].values

    fpr, tpr, _ = roc_curve(binary, scores)

    auc = roc_auc_score(binary, scores)

    ax.plot(
        fpr,
        tpr,
        linewidth=2,
        label=f"{cls} (AUC={auc:.3f})"
    )

ax.plot(
    [0, 1],
    [0, 1],
    "k--",
    linewidth=1.5
)

ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("SVM LOCO ROC — CLR Only")

ax.legend(frameon=False)

ax.grid(False)

for spine in ax.spines.values():
    spine.set_linewidth(1.5)

plt.tight_layout()

plt.savefig(
    f"{OUTPUT_DIR}/LOCO_CLRonly_ROC.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

# ================================================================
# 9. COMPARE WITH PREVIOUS BATCH-CORRECTED LOCO
# ================================================================

previous = {
    "Accuracy": 0.48643216080402,
    "BalancedAccuracy": 0.429951925108071,
    "MacroF1": 0.418379526987718,
    "MCC": 0.175747723815669,
    "AUROC_Healthy": 0.666159012808497,
    "AUROC_Adenoma": 0.575007215007215,
    "AUROC_Cancer": 0.700381123464582
}

comparison = []

for metric, old_value in previous.items():

    new_value = float(pooled_df[metric].iloc[0])

    comparison.append({
        "Metric": metric,
        "CLR_BatchCorrected": old_value,
        "CLR_Only": new_value,
        "Difference_CLRonly_minus_corrected":
            new_value - old_value
    })

comparison_df = pd.DataFrame(comparison)

comparison_df.to_csv(
    f"{OUTPUT_DIR}/LOCO_Preprocessing_Comparison.csv",
    index=False
)

print("\n======================================")
print("PREPROCESSING COMPARISON")
print("======================================")

print(
    comparison_df.round(4).to_string(index=False)
)

print("\nAll outputs saved in:", OUTPUT_DIR)
print("CLR-only LOCO analysis completed.")
